In [1]:
# -*- coding: utf-8 -*-
"""
通用 PINN 二群/多材料基准题验证脚本

用法：
1) 修改 MODEL_DEF_PY 指向训练脚本 1.py；1.py 中至少应包含：
   - MultiMatPINN
   - get_material_id
   - 训练时使用的裂变截面/材料参数，例如 NU_SIGMA_F1、NU_SIGMA_F2，
     或 MATERIALS / MATERIAL_DATA 这类按材料编号存储的字典。
2) 修改 MODEL_PATH、CSV_PATH 指向当前基准题的权重和参考解 CSV。
3) 运行本脚本。

相比原 BIBLIS 专用脚本，本脚本不再硬编码：
- 几何长度 L、外边界 EXT_SEGMENTS
- 材料编号 1~8
- BIBLIS 的燃料/反射层划分
- BIBLIS 的 NU_SIGMA_F1 / NU_SIGMA_F2
- BIBLIS 对角线端点

这些信息会优先从 1.py 和 CSV 点集自动推断。
"""

import os
import re
import math
import inspect
import importlib.util
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from matplotlib import rcParams

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
rcParams["font.sans-serif"] = ["SimHei", "Microsoft YaHei", "SimSun", "Arial Unicode MS"]
rcParams["axes.unicode_minus"] = False

# =========================
# 0) 配置：通常只需要改这里
# =========================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 训练脚本：从这里读取网络结构、几何判定 get_material_id、材料参数等
MODEL_DEF_PY = r"C:\Users\Administrator\Downloads\1.py"

# 当前基准题对应的权重文件和参考解 CSV
MODEL_PATH = r"C:\Users\Administrator\Downloads\RTXpinn.pth"
CSV_PATH = r"C:\Users\Administrator\Desktop\Comsol实验\TWIGL\TWIGL.csv"

# None 时自动取权重或 CSV 文件名；也可手动写 "BIBLIS"、"IAEA2D" 等
CASE_NAME = None

# CSV 读取设置
# CSV_HEADER = "infer"：第一行为表头；CSV_HEADER = None：无表头
CSV_HEADER = "infer"

# "auto" 会优先按列名识别 x,y,phi1,phi2；识别失败再按索引读取
CSV_USE_COLNAMES = "auto"  # 可选："auto" / True / False
CSV_COLS_BY_INDEX = dict(x=0, y=1, phi1=3, phi2=4)
CSV_COLS_BY_NAME = dict(x="x", y="y", phi1="phi1", phi2="phi2")

# 是否做裂变源归一化：需要能从 1.py 中读取各材料的 nu_sigma_f1/2
NORMALIZE_BY_FISSION_SOURCE = True

# 相对误差分母保护：rel = |pred-ref| / (|ref| + REL_EPS_FACTOR * max|ref|)
REL_EPS_FACTOR = 0.1

# 可视化裁剪；None 表示不裁剪
REL_ERR_CLIP = 1.0
ABS_ERR_CLIP = None

# 是否画自动剖面线。None 表示取参考点有效区域外接矩形的反对角线：(xmin,ymax)->(xmax,ymin)
PLOT_PROFILE = True
PROFILE_LINE = None  # 例如 ((0.0, 196.5421), (196.5421, 0.0))
PROFILE_N = 700

SAVE_FIG = True
OUT_DIR = "./eval_plots"
SAVE_METRICS_CSV = True

# 边界材料判定容差。若 1.py 的 get_material_id 在边界点返回 0，本脚本会向周围探测非零材料号。
BOUNDARY_TOL = 1e-6
BOUNDARY_EPS_FACTOR = 1e-7   # eps = 坐标尺度 * 该因子
BOUNDARY_EPS_MIN = 1e-6
BOUNDARY_PROBE_RINGS = (1.0, 2.0, 5.0, 10.0, 50.0)

# 将裂变截面视为 0 的阈值，用于自动区分 fuel / non_fuel
FISSION_EPS = 1e-20


# =========================
# 1) 动态导入训练脚本
# =========================
def load_train_module(py_path: str):
    py_path = str(Path(py_path).resolve())
    spec = importlib.util.spec_from_file_location("train_model_def", py_path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"无法从 {py_path} 导入模块")
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


def optional_build_interface_lines(mod: Any, device: torch.device):
    """若 1.py 中存在 build_true_interface_lines，则尽量按其签名调用。"""
    if not hasattr(mod, "build_true_interface_lines"):
        return

    fn = mod.build_true_interface_lines
    kwargs = {}
    try:
        sig = inspect.signature(fn)
        if "eps" in sig.parameters:
            kwargs["eps"] = 1e-2
        if "n_scan" in sig.parameters:
            kwargs["n_scan"] = 512
        if "device" in sig.parameters:
            kwargs["device"] = device
    except Exception:
        kwargs = dict(eps=1e-2, n_scan=512, device=device)

    try:
        val = fn(**kwargs)
        if isinstance(val, tuple) and len(val) >= 2:
            mod.TRUE_X_IF, mod.TRUE_Y_IF = val[0], val[1]
        print("[信息] 已从 1.py 构建/更新 TRUE_X_IF、TRUE_Y_IF。")
    except Exception as e:
        print(f"[提示] build_true_interface_lines 调用失败，继续评估：{e}")


# =========================
# 2) 通用材料判定：边界点自动往域内探测
# =========================
def _as_col_tensor(v: torch.Tensor) -> torch.Tensor:
    return v if v.dim() == 2 else v.view(-1, 1)


def _call_get_material_id(get_material_id_fn: Any, x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """兼容 get_material_id(x,y) 和 get_material_id(xy) 两种写法。"""
    x = _as_col_tensor(x)
    y = _as_col_tensor(y)
    xy = torch.cat([x, y], dim=1)

    try:
        sig = inspect.signature(get_material_id_fn)
        n_pos = sum(
            1 for p in sig.parameters.values()
            if p.kind in (p.POSITIONAL_ONLY, p.POSITIONAL_OR_KEYWORD)
            and p.default is p.empty
        )
        if n_pos <= 1:
            return get_material_id_fn(xy)
        return get_material_id_fn(x, y)
    except Exception:
        # 若无法解析签名，按训练脚本最常见的 x,y 方式先试
        try:
            return get_material_id_fn(x, y)
        except TypeError:
            return get_material_id_fn(xy)


def _coord_scale(x: torch.Tensor, y: torch.Tensor) -> float:
    xy = torch.cat([_as_col_tensor(x), _as_col_tensor(y)], dim=1)
    finite = torch.isfinite(xy)
    if not bool(finite.any()):
        return 1.0
    mn = float(torch.min(xy[finite]).detach().cpu())
    mx = float(torch.max(xy[finite]).detach().cpu())
    return max(abs(mx - mn), abs(mx), abs(mn), 1.0)


def get_material_id_eval(
    get_material_id_fn: Any,
    x_coords: torch.Tensor,
    y_coords: torch.Tensor,
    eps: Optional[float] = None,
    tol: float = BOUNDARY_TOL,
    probe_rings: Sequence[float] = BOUNDARY_PROBE_RINGS,
) -> torch.Tensor:
    """
    通用评估材料号函数。

    原 BIBLIS 脚本硬编码了 x=0/y=0 镜像边界和顶边/右边外边界的内移规则。
    这里改成：先直接调用 1.py 的 get_material_id；若返回 0，则在该点附近多方向小步探测，
    取第一个非零材料号。这样几何边界不需要在评估脚本里重写。
    """
    x = _as_col_tensor(x_coords)
    y = _as_col_tensor(y_coords)

    if eps is None:
        eps = max(BOUNDARY_EPS_MIN, _coord_scale(x, y) * BOUNDARY_EPS_FACTOR)

    mat = _call_get_material_id(get_material_id_fn, x, y).view(-1).long()
    bad = (mat == 0) | (~torch.isfinite(mat.float()))
    if not bool(bad.any()):
        return mat

    # 16 个方向，避免只适用于矩形边界
    directions = [
        (1.0, 0.0), (-1.0, 0.0), (0.0, 1.0), (0.0, -1.0),
        (1.0, 1.0), (1.0, -1.0), (-1.0, 1.0), (-1.0, -1.0),
        (2.0, 1.0), (2.0, -1.0), (-2.0, 1.0), (-2.0, -1.0),
        (1.0, 2.0), (1.0, -2.0), (-1.0, 2.0), (-1.0, -2.0),
    ]

    mat2 = mat.clone()
    for ring in probe_rings:
        if not bool((mat2 == 0).any()):
            break
        step = eps * float(ring)
        for dx, dy in directions:
            norm = math.sqrt(dx * dx + dy * dy)
            dxn, dyn = step * dx / norm, step * dy / norm
            cand = _call_get_material_id(get_material_id_fn, x + dxn, y + dyn).view(-1).long()
            pick = (mat2 == 0) & (cand != 0)
            mat2[pick] = cand[pick]
            if not bool((mat2 == 0).any()):
                break

    return mat2


# =========================
# 3) 从 1.py 读取材料裂变参数
# =========================
def _is_numeric_scalar(x: Any) -> bool:
    try:
        if isinstance(x, torch.Tensor):
            return x.numel() == 1
        arr = np.asarray(x)
        return arr.size == 1 and np.issubdtype(arr.dtype, np.number)
    except Exception:
        return False


def _to_float(x: Any) -> float:
    if isinstance(x, torch.Tensor):
        return float(x.detach().cpu().reshape(-1)[0])
    return float(np.asarray(x).reshape(-1)[0])


def _is_numeric_container(x: Any) -> bool:
    if isinstance(x, dict):
        return len(x) > 0
    try:
        arr = np.asarray(x)
        return arr.ndim >= 1 and arr.size > 0 and np.issubdtype(arr.dtype, np.number)
    except Exception:
        return False


def _container_to_mat_map(obj: Any, material_ids: Sequence[int]) -> Optional[Dict[int, float]]:
    """把 dict/list/np.ndarray/tensor 转成 {材料号: 数值}。"""
    ids = [int(i) for i in material_ids]
    if not ids:
        return None

    if isinstance(obj, dict):
        out = {}
        for mid in ids:
            keys = [mid, str(mid), f"mat{mid}", f"material{mid}", f"MAT{mid}"]
            found = False
            for k in keys:
                if k in obj and _is_numeric_scalar(obj[k]):
                    out[mid] = _to_float(obj[k])
                    found = True
                    break
            if not found:
                return None
        return out

    try:
        arr = np.asarray(obj.detach().cpu() if isinstance(obj, torch.Tensor) else obj, dtype=float).reshape(-1)
    except Exception:
        return None

    max_id = max(ids)
    out = {}
    # 优先支持 1-based 且数组第 0 位闲置：arr[mat_id]
    if len(arr) > max_id:
        for mid in ids:
            out[mid] = float(arr[mid])
        return out
    # 支持紧凑 1-based：arr[mat_id-1]
    if len(arr) >= max_id:
        for mid in ids:
            out[mid] = float(arr[mid - 1])
        return out
    return None


def _find_attr_case_insensitive(obj: Any, names: Iterable[str]) -> Optional[Any]:
    attrs = vars(obj)
    lower_map = {k.lower(): k for k in attrs.keys()}
    for name in names:
        k = lower_map.get(name.lower())
        if k is not None:
            return attrs[k]
    return None


def _attr_name_candidates_for_group(group: int) -> List[str]:
    g = str(group)
    return [
        f"NU_SIGMA_F{g}", f"NU_SIGMAF{g}", f"NUSIGMAF{g}", f"NUSIGF{g}",
        f"nu_sigma_f{g}", f"nuSigmaF{g}", f"nu_sig_f{g}",
        f"NU_SIGMA_F_G{g}", f"nu_sigma_f_g{g}",
        f"NUFISS{g}", f"nu_fission{g}", f"fission{g}",
    ]


def _generic_find_group_container(mod: Any, group: int) -> Optional[Any]:
    attrs = vars(mod)
    # 名字里必须像“nu sigma f / nusigmaf / fission”，并能看出 group 1/2
    group_tokens = [str(group), f"g{group}", f"group{group}", f"grp{group}"]
    scored = []
    for name, val in attrs.items():
        low = name.lower().replace("_", "")
        has_fission = (
            ("nu" in low and "sigma" in low and "f" in low)
            or "nusigmaf" in low
            or "nusigf" in low
            or "fission" in low
        )
        has_group = any(tok in low for tok in group_tokens)
        if has_fission and has_group and _is_numeric_container(val):
            # 越短越可能是精确变量名
            scored.append((len(low), name, val))
    if not scored:
        return None
    scored.sort(key=lambda t: t[0])
    return scored[0][2]


def _extract_from_material_dict(mod: Any, material_ids: Sequence[int]) -> Optional[Tuple[Dict[int, float], Dict[int, float]]]:
    """兼容 MATERIALS = {1: {'nu_sigma_f1':..., 'nu_sigma_f2':...}, ...} 这类写法。"""
    attrs = vars(mod)
    key1_patterns = ["nu_sigma_f1", "nusigmaf1", "nusigf1", "nuSigmaF1", "nu_fission1", "fission1"]
    key2_patterns = ["nu_sigma_f2", "nusigmaf2", "nusigf2", "nuSigmaF2", "nu_fission2", "fission2"]

    for name, obj in attrs.items():
        if not isinstance(obj, dict):
            continue
        low_name = name.lower()
        if not any(tok in low_name for tok in ["mat", "xs", "cross", "sigma", "material", "region"]):
            continue

        nu1, nu2 = {}, {}
        ok = True
        for mid in material_ids:
            mid = int(mid)
            entry = None
            for k in [mid, str(mid), f"mat{mid}", f"material{mid}", f"MAT{mid}"]:
                if k in obj:
                    entry = obj[k]
                    break
            if entry is None:
                ok = False
                break

            if isinstance(entry, dict):
                entry_lower = {str(k).lower().replace("_", ""): v for k, v in entry.items()}
                v1 = None
                v2 = None
                for pat in key1_patterns:
                    kk = pat.lower().replace("_", "")
                    if kk in entry_lower:
                        v1 = entry_lower[kk]
                        break
                for pat in key2_patterns:
                    kk = pat.lower().replace("_", "")
                    if kk in entry_lower:
                        v2 = entry_lower[kk]
                        break
                if v1 is None or v2 is None or not _is_numeric_scalar(v1) or not _is_numeric_scalar(v2):
                    ok = False
                    break
                nu1[mid] = _to_float(v1)
                nu2[mid] = _to_float(v2)
            elif isinstance(entry, (list, tuple, np.ndarray, torch.Tensor)):
                # 尝试把 entry 当作至少含两个裂变项的序列；此分支无法保证语义，放到最后
                arr = np.asarray(entry.detach().cpu() if isinstance(entry, torch.Tensor) else entry, dtype=float).reshape(-1)
                if arr.size < 2:
                    ok = False
                    break
                # 约定：最后两个或前两个均可能，这里只在变量名强相关时才接受前两个
                nu1[mid] = float(arr[0])
                nu2[mid] = float(arr[1])
            else:
                ok = False
                break

        if ok and nu1 and nu2:
            print(f"[信息] 已从 1.py 的 {name} 中读取裂变参数。")
            return nu1, nu2

    return None


def extract_nu_sigma_f_maps(mod: Any, material_ids: Sequence[int]) -> Tuple[Dict[int, float], Dict[int, float]]:
    """
    从 1.py 读取各材料的 nu_sigma_f1 / nu_sigma_f2。
    支持：
    - NU_SIGMA_F1 = {1:..., 2:...}; NU_SIGMA_F2 = {...}
    - NU_SIGMA_F1 / NU_SIGMA_F2 为 list/np.ndarray/tensor
    - MATERIALS / MATERIAL_DATA 这类嵌套字典
    """
    ids = [int(i) for i in material_ids]

    obj1 = _find_attr_case_insensitive(mod, _attr_name_candidates_for_group(1))
    obj2 = _find_attr_case_insensitive(mod, _attr_name_candidates_for_group(2))

    if obj1 is None:
        obj1 = _generic_find_group_container(mod, 1)
    if obj2 is None:
        obj2 = _generic_find_group_container(mod, 2)

    if obj1 is not None and obj2 is not None:
        m1 = _container_to_mat_map(obj1, ids)
        m2 = _container_to_mat_map(obj2, ids)
        if m1 is not None and m2 is not None:
            print("[信息] 已从 1.py 的 NU_SIGMA_F1/NU_SIGMA_F2 类变量中读取裂变参数。")
            return m1, m2

    nested = _extract_from_material_dict(mod, ids)
    if nested is not None:
        return nested

    raise RuntimeError(
        "无法从 1.py 自动读取各材料的 nu_sigma_f1 / nu_sigma_f2。\n"
        "请在 1.py 中提供类似以下变量之一：\n"
        "  NU_SIGMA_F1 = {1: ..., 2: ..., ...}\n"
        "  NU_SIGMA_F2 = {1: ..., 2: ..., ...}\n"
        "或 MATERIALS = {1: {'nu_sigma_f1': ..., 'nu_sigma_f2': ...}, ...}\n"
        "若不需要裂变源归一化，可把 NORMALIZE_BY_FISSION_SOURCE = False。"
    )


def nu_sigma_f_from_mat(
    mat_ids_np: np.ndarray,
    nu1_map: Dict[int, float],
    nu2_map: Dict[int, float],
) -> Tuple[np.ndarray, np.ndarray]:
    mat = np.asarray(mat_ids_np, dtype=np.int64).reshape(-1)
    nu1 = np.zeros_like(mat, dtype=np.float32)
    nu2 = np.zeros_like(mat, dtype=np.float32)
    for mid in np.unique(mat):
        mid = int(mid)
        if mid <= 0:
            continue
        if mid not in nu1_map or mid not in nu2_map:
            raise RuntimeError(f"材料 {mid} 缺少 nu_sigma_f1/2 参数。")
        nu1[mat == mid] = float(nu1_map[mid])
        nu2[mat == mid] = float(nu2_map[mid])
    return nu1, nu2


# =========================
# 4) 模型调用、CSV 读取、误差指标
# =========================
def instantiate_model(mod: Any) -> torch.nn.Module:
    if not hasattr(mod, "MultiMatPINN"):
        raise RuntimeError("1.py 中未找到 MultiMatPINN 类。")
    return mod.MultiMatPINN().to(DEVICE)


def call_model(model: torch.nn.Module, xy_t: torch.Tensor, mat_ids_t: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    """兼容 model(xy, mat_ids) / model(xy) 以及 tuple 或 Nx2 tensor 输出。"""
    try:
        out = model(xy_t, mat_ids_t)
    except TypeError:
        out = model(xy_t)

    if isinstance(out, (tuple, list)):
        if len(out) < 2:
            raise RuntimeError("模型输出 tuple/list 长度小于 2，无法取得 phi1/phi2。")
        phi1_t, phi2_t = out[0], out[1]
    else:
        if out.dim() != 2 or out.shape[1] < 2:
            raise RuntimeError("模型输出既不是 (phi1, phi2, ...) 也不是 Nx2/Nx>=2 tensor。")
        phi1_t, phi2_t = out[:, 0:1], out[:, 1:2]

    return phi1_t, phi2_t


def load_state_dict_flexible(model: torch.nn.Module, model_path: str, device: torch.device):
    obj = torch.load(model_path, map_location=device)
    if isinstance(obj, dict):
        for key in ["model", "state_dict", "model_state_dict", "net", "network"]:
            if key in obj and isinstance(obj[key], dict):
                obj = obj[key]
                break
    missing, unexpected = model.load_state_dict(obj, strict=False)
    if missing or unexpected:
        print("[提示] load_state_dict 非严格匹配：")
        if missing:
            print("  missing keys:", missing[:10], "..." if len(missing) > 10 else "")
        if unexpected:
            print("  unexpected keys:", unexpected[:10], "..." if len(unexpected) > 10 else "")


def _normalize_col_name(s: str) -> str:
    return str(s).strip().lower().replace(" ", "").replace("-", "_")


def _find_col_by_alias(df: pd.DataFrame, aliases: Sequence[str]) -> Optional[str]:
    norm_to_col = {_normalize_col_name(c): c for c in df.columns}
    alias_norm = [_normalize_col_name(a) for a in aliases]
    for a in alias_norm:
        if a in norm_to_col:
            return norm_to_col[a]
    for norm, col in norm_to_col.items():
        if any(a in norm for a in alias_norm):
            return col
    return None


def read_reference_csv(csv_path: str):
    header = None if CSV_HEADER is None else "infer"
    df = pd.read_csv(csv_path, header=header)

    use_names = CSV_USE_COLNAMES
    if use_names == "auto":
        x_col = _find_col_by_alias(df, [CSV_COLS_BY_NAME.get("x", "x"), "x", "coord_x", "x_cm"])
        y_col = _find_col_by_alias(df, [CSV_COLS_BY_NAME.get("y", "y"), "y", "coord_y", "y_cm"])
        p1_col = _find_col_by_alias(df, [CSV_COLS_BY_NAME.get("phi1", "phi1"), "phi_1", "flux1", "flux_1", "group1", "g1"])
        p2_col = _find_col_by_alias(df, [CSV_COLS_BY_NAME.get("phi2", "phi2"), "phi_2", "flux2", "flux_2", "group2", "g2"])
        use_names = all(c is not None for c in [x_col, y_col, p1_col, p2_col])
    elif use_names:
        x_col = CSV_COLS_BY_NAME["x"]
        y_col = CSV_COLS_BY_NAME["y"]
        p1_col = CSV_COLS_BY_NAME["phi1"]
        p2_col = CSV_COLS_BY_NAME["phi2"]
    else:
        x_col = y_col = p1_col = p2_col = None

    if use_names:
        x = df[x_col].to_numpy(np.float32)
        y = df[y_col].to_numpy(np.float32)
        phi1_ref = df[p1_col].to_numpy(np.float32)
        phi2_ref = df[p2_col].to_numpy(np.float32)
        print(f"[信息] CSV 按列名读取：x={x_col}, y={y_col}, phi1={p1_col}, phi2={p2_col}")
    else:
        x = df.iloc[:, CSV_COLS_BY_INDEX["x"]].to_numpy(np.float32)
        y = df.iloc[:, CSV_COLS_BY_INDEX["y"]].to_numpy(np.float32)
        phi1_ref = df.iloc[:, CSV_COLS_BY_INDEX["phi1"]].to_numpy(np.float32)
        phi2_ref = df.iloc[:, CSV_COLS_BY_INDEX["phi2"]].to_numpy(np.float32)
        print(f"[信息] CSV 按列索引读取：{CSV_COLS_BY_INDEX}")

    return x, y, phi1_ref, phi2_ref


def compute_metrics(ref: np.ndarray, pred: np.ndarray, rel_eps: float) -> Dict[str, float]:
    ref = np.asarray(ref).reshape(-1)
    pred = np.asarray(pred).reshape(-1)
    ok = np.isfinite(ref) & np.isfinite(pred)
    ref = ref[ok]
    pred = pred[ok]
    if ref.size == 0:
        return {"L2_rel": np.nan, "MSE": np.nan, "RMSE": np.nan, "MAE": np.nan,
                "MaxRelErr": np.nan, "MeanRelErr": np.nan, "R2": np.nan}

    diff = pred - ref
    mse = float(np.mean(diff ** 2))
    rmse = float(math.sqrt(mse))
    mae = float(np.mean(np.abs(diff)))
    l2_num = float(np.linalg.norm(diff, ord=2))
    l2_den = float(np.linalg.norm(ref, ord=2) + 1e-30)
    l2_rel = l2_num / l2_den
    rel = np.abs(diff) / (np.abs(ref) + rel_eps)
    max_rel = float(np.max(rel))
    mean_rel = float(np.mean(rel))
    sse = float(np.sum(diff ** 2))
    ref_mean = float(np.mean(ref))
    sst = float(np.sum((ref - ref_mean) ** 2))
    r2 = float("nan") if sst < 1e-30 else (1.0 - sse / (sst + 1e-30))
    return {"L2_rel": l2_rel, "MSE": mse, "RMSE": rmse, "MAE": mae,
            "MaxRelErr": max_rel, "MeanRelErr": mean_rel, "R2": r2}


def build_region_masks(
    mat_ids_np: np.ndarray,
    nu1_map: Optional[Dict[int, float]] = None,
    nu2_map: Optional[Dict[int, float]] = None,
):
    mat = np.asarray(mat_ids_np, dtype=np.int64).reshape(-1)
    valid_ids = sorted(int(i) for i in np.unique(mat) if int(i) > 0)
    masks = {f"mat{mid}": (mat == mid) for mid in valid_ids}
    masks["all"] = mat > 0

    ordered = [f"mat{mid}" for mid in valid_ids]

    if nu1_map is not None and nu2_map is not None:
        fuel_ids = [mid for mid in valid_ids if abs(nu1_map.get(mid, 0.0)) + abs(nu2_map.get(mid, 0.0)) > FISSION_EPS]
        non_fuel_ids = [mid for mid in valid_ids if mid not in fuel_ids]
        if fuel_ids:
            masks["fuel"] = np.isin(mat, fuel_ids)
            ordered.append("fuel")
        if non_fuel_ids:
            masks["non_fuel"] = np.isin(mat, non_fuel_ids)
            ordered.append("non_fuel")

    ordered.append("all")
    return masks, ordered, valid_ids


# =========================
# 5) 剖面线：参考解散点插值 + 模型沿线预测
# =========================
def make_line_points(p0, p1, n=600):
    x0, y0 = p0
    x1, y1 = p1
    t = np.linspace(0.0, 1.0, n, dtype=np.float32)
    x = x0 + (x1 - x0) * t
    y = y0 + (y1 - y0) * t
    s = np.sqrt((x - x0) ** 2 + (y - y0) ** 2).astype(np.float32)
    return x, y, s


def idw_interp_scattered(x_ref, y_ref, v_ref, xq, yq, k=8, power=2.0, chunk=256):
    pts = np.stack([x_ref, y_ref], axis=1).astype(np.float64)
    vals = np.asarray(v_ref, dtype=np.float64).reshape(-1)
    q = np.stack([xq, yq], axis=1).astype(np.float64)

    n_ref = pts.shape[0]
    k = max(1, min(int(k), n_ref))
    out = np.empty(len(q), dtype=np.float64)

    for i in range(0, len(q), chunk):
        qq = q[i:i + chunk]
        d2 = ((qq[:, None, 0] - pts[None, :, 0]) ** 2 +
              (qq[:, None, 1] - pts[None, :, 1]) ** 2)

        idx = np.argpartition(d2, kth=k - 1, axis=1)[:, :k]
        d2_k = np.take_along_axis(d2, idx, axis=1)
        v_k = vals[idx]

        exact = d2_k <= 1e-24
        w = 1.0 / np.maximum(d2_k, 1e-24) ** (power / 2.0)
        out_chunk = np.sum(w * v_k, axis=1) / np.sum(w, axis=1)

        rows = np.where(np.any(exact, axis=1))[0]
        for r in rows:
            j = np.argmin(d2_k[r])
            out_chunk[r] = v_k[r, j]

        out[i:i + chunk] = out_chunk

    return out.astype(np.float32)


@torch.no_grad()
def predict_profile_line(mod, model, p0, p1, device, n=600):
    x_line, y_line, s_line = make_line_points(p0, p1, n=n)

    xy_t = torch.from_numpy(np.stack([x_line, y_line], axis=1).astype(np.float32)).to(device)
    x_t = xy_t[:, 0:1]
    y_t = xy_t[:, 1:2]

    mat_ids_t = get_material_id_eval(mod.get_material_id, x_t, y_t).long().view(-1)
    mat_ids = mat_ids_t.detach().cpu().numpy().astype(np.int64)
    valid = mat_ids > 0

    phi1_pred = np.full(len(x_line), np.nan, dtype=np.float32)
    phi2_pred = np.full(len(x_line), np.nan, dtype=np.float32)

    if np.any(valid):
        phi1_t, phi2_t = call_model(model, xy_t[valid], mat_ids_t[valid])
        phi1_pred[valid] = phi1_t.squeeze(-1).detach().cpu().numpy().astype(np.float32)
        phi2_pred[valid] = phi2_t.squeeze(-1).detach().cpu().numpy().astype(np.float32)

    return x_line, y_line, s_line, mat_ids, phi1_pred, phi2_pred


def find_interface_positions_on_line(s_line, mat_ids):
    mats = np.asarray(mat_ids).reshape(-1)
    valid_pair = (mats[:-1] > 0) & (mats[1:] > 0)
    jump = valid_pair & (mats[:-1] != mats[1:])
    idx = np.where(jump)[0]
    return [0.5 * (s_line[i] + s_line[i + 1]) for i in idx]


def infer_profile_line_from_points(x, y, in_domain):
    if PROFILE_LINE is not None:
        return PROFILE_LINE
    xv = np.asarray(x)[in_domain]
    yv = np.asarray(y)[in_domain]
    if xv.size == 0:
        xv = np.asarray(x)
        yv = np.asarray(y)
    xmin, xmax = float(np.nanmin(xv)), float(np.nanmax(xv))
    ymin, ymax = float(np.nanmin(yv)), float(np.nanmax(yv))
    return (xmin, ymax), (xmax, ymin)


def plot_profile_line(
    mod, model,
    x_ref_all, y_ref_all, phi1_ref_all, phi2_ref_all,
    p0, p1,
    S_ref, scale,
    case_name="case",
    save_fig=True, out_dir="./eval_plots",
    n_line=600,
):
    x_line, y_line, s_line, mat_line, phi1_pred, phi2_pred = predict_profile_line(
        mod, model, p0, p1, DEVICE, n=n_line
    )

    phi1_ref_line = idw_interp_scattered(
        x_ref_all, y_ref_all, phi1_ref_all, x_line, y_line, k=8, power=2.0
    )
    phi2_ref_line = idw_interp_scattered(
        x_ref_all, y_ref_all, phi2_ref_all, x_line, y_line, k=8, power=2.0
    )

    eps = 1e-30
    phi1_ref_line = phi1_ref_line / (S_ref + eps)
    phi2_ref_line = phi2_ref_line / (S_ref + eps)
    phi1_pred = phi1_pred * scale / (S_ref + eps)
    phi2_pred = phi2_pred * scale / (S_ref + eps)

    valid = mat_line > 0
    phi1_ref_line[~valid] = np.nan
    phi2_ref_line[~valid] = np.nan
    phi1_pred[~valid] = np.nan
    phi2_pred[~valid] = np.nan

    interface_s = find_interface_positions_on_line(s_line, mat_line)

    fig, axes = plt.subplots(2, 1, figsize=(8.5, 6.8), sharex=True)

    axes[0].plot(s_line, phi1_ref_line, label="Reference", linewidth=2.0)
    axes[0].plot(s_line, phi1_pred, label="Prediction", linewidth=1.6)
    axes[0].set_ylabel(r"$\phi_1$")
    axes[0].set_title(f"{case_name} profile line: group 1")

    axes[1].plot(s_line, phi2_ref_line, label="Reference", linewidth=2.0)
    axes[1].plot(s_line, phi2_pred, label="Prediction", linewidth=1.6)
    axes[1].set_ylabel(r"$\phi_2$")
    axes[1].set_xlabel("Distance along the profile line")
    axes[1].set_title(f"{case_name} profile line: group 2")

    for ax in axes:
        for ss in interface_s:
            ax.axvline(ss, linestyle="--", linewidth=1.0, alpha=0.6)
        ax.grid(True, alpha=0.25)
        ax.legend()

    plt.tight_layout()

    if save_fig:
        os.makedirs(out_dir, exist_ok=True)
        fname = f"{_safe_name(case_name)}_profile_line.png"
        out_path = os.path.join(out_dir, fname)
        plt.savefig(out_path, dpi=300, bbox_inches="tight")
        print(f"[已保存] 剖面线图：{out_path}")

    plt.show()


# =========================
# 6) 可视化
# =========================
def _safe_name(name: str) -> str:
    return re.sub(r"[^0-9a-zA-Z_\-]+", "_", str(name)).strip("_") or "case"


def clip_sym(arr, clip_val):
    if clip_val is None:
        return arr, None
    return np.clip(arr, -clip_val, clip_val), clip_val


def finite_abs_max(arr: np.ndarray, mask: np.ndarray) -> float:
    vals = np.asarray(arr)[mask]
    vals = vals[np.isfinite(vals)]
    if vals.size == 0:
        return 1.0
    vmax = float(np.max(np.abs(vals)))
    return vmax if vmax > 0 else 1.0


def plot_global_error_maps(
    x, y, in_domain,
    phi1_ref_n, phi2_ref_n,
    phi1_pred_n, phi2_pred_n,
    rel_eps_1, rel_eps_2,
    case_name,
):
    err1 = phi1_pred_n - phi1_ref_n
    err2 = phi2_pred_n - phi2_ref_n
    rel_err1 = err1 / (np.abs(phi1_ref_n) + rel_eps_1)
    rel_err2 = err2 / (np.abs(phi2_ref_n) + rel_eps_2)

    err1_p, _ = clip_sym(err1, ABS_ERR_CLIP)
    err2_p, _ = clip_sym(err2, ABS_ERR_CLIP)
    rel1_p, v1 = clip_sym(rel_err1, REL_ERR_CLIP)
    rel2_p, v2 = clip_sym(rel_err2, REL_ERR_CLIP)

    mplot = in_domain & np.isfinite(x) & np.isfinite(y)
    x_p, y_p = x[mplot], y[mplot]

    vmax_err1 = finite_abs_max(err1_p, mplot)
    vmax_err2 = finite_abs_max(err2_p, mplot)

    fig, axes = plt.subplots(2, 2, figsize=(12, 10))

    ax = axes[0, 0]
    sc = ax.scatter(x_p, y_p, c=err1_p[mplot], s=10, marker="s", cmap="seismic",
                    vmin=-vmax_err1, vmax=vmax_err1)
    ax.set_title("归一化后 φ1 误差")
    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    plt.colorbar(sc, ax=ax)

    ax = axes[0, 1]
    sc = ax.scatter(x_p, y_p, c=rel1_p[mplot], s=10, marker="s", cmap="seismic",
                    vmin=(-v1 if v1 is not None else None),
                    vmax=(v1 if v1 is not None else None))
    ax.set_title("φ1 的相对误差")
    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    plt.colorbar(sc, ax=ax)

    ax = axes[1, 0]
    sc = ax.scatter(x_p, y_p, c=err2_p[mplot], s=10, marker="s", cmap="seismic",
                    vmin=-vmax_err2, vmax=vmax_err2)
    ax.set_title("归一化后 φ2 误差")
    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    plt.colorbar(sc, ax=ax)

    ax = axes[1, 1]
    sc = ax.scatter(x_p, y_p, c=rel2_p[mplot], s=10, marker="s", cmap="seismic",
                    vmin=(-v2 if v2 is not None else None),
                    vmax=(v2 if v2 is not None else None))
    ax.set_title("φ2 的相对误差")
    ax.set_aspect("equal")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    plt.colorbar(sc, ax=ax)

    plt.tight_layout()

    if SAVE_FIG:
        os.makedirs(OUT_DIR, exist_ok=True)
        out_path = os.path.join(OUT_DIR, f"{_safe_name(case_name)}_global_error_maps.png")
        plt.savefig(out_path, dpi=200, bbox_inches="tight")
        print(f"[已保存] 全域偏差分布图：{out_path}")

    plt.show()


# =========================
# 7) 主流程
# =========================
def main():
    case_name = CASE_NAME
    if case_name is None:
        stem = Path(MODEL_PATH).stem
        case_name = stem if stem.lower() not in ["model", "best", "checkpoint"] else Path(CSV_PATH).stem

    # ---- 7.1 导入训练脚本模块 ----
    mod = load_train_module(MODEL_DEF_PY)
    if not hasattr(mod, "get_material_id"):
        raise RuntimeError("1.py 中未找到 get_material_id。")
    optional_build_interface_lines(mod, DEVICE)

    # ---- 7.2 实例化模型并加载权重 ----
    model = instantiate_model(mod)
    load_state_dict_flexible(model, MODEL_PATH, DEVICE)
    model.eval()
    print("[信息] 模型加载完成。")

    # ---- 7.3 读取 CSV ----
    x, y, phi1_ref, phi2_ref = read_reference_csv(CSV_PATH)
    print(f"[信息] CSV 读取完成，共 {len(x)} 个点。")

    # ---- 7.4 模型预测 ----
    xy = np.stack([x, y], axis=1).astype(np.float32)
    xy_t = torch.from_numpy(xy).to(DEVICE)

    with torch.no_grad():
        x_t = xy_t[:, 0:1]
        y_t = xy_t[:, 1:2]
        mat_ids_t = get_material_id_eval(mod.get_material_id, x_t, y_t).long().view(-1)
        phi1_pred_t, phi2_pred_t = call_model(model, xy_t, mat_ids_t)

    phi1_pred = phi1_pred_t.squeeze(-1).detach().cpu().numpy().astype(np.float32)
    phi2_pred = phi2_pred_t.squeeze(-1).detach().cpu().numpy().astype(np.float32)
    mat_ids_np = mat_ids_t.detach().cpu().numpy().astype(np.int64)

    in_domain = mat_ids_np > 0
    if not np.any(in_domain):
        raise RuntimeError("所有参考点的材料号均为 0。请检查 1.py 的 get_material_id 与 CSV 坐标单位/范围是否一致。")

    material_ids = sorted(int(i) for i in np.unique(mat_ids_np[in_domain]))
    print(f"[信息] 自动识别到材料编号：{material_ids}")

    # ---- 7.5 从 1.py 读取裂变参数，并做裂变源归一化 ----
    nu1_map = nu2_map = None
    if NORMALIZE_BY_FISSION_SOURCE:
        nu1_map, nu2_map = extract_nu_sigma_f_maps(mod, material_ids)
        nu1, nu2 = nu_sigma_f_from_mat(mat_ids_np, nu1_map, nu2_map)

        S_ref = float(np.sum(nu1[in_domain] * phi1_ref[in_domain] + nu2[in_domain] * phi2_ref[in_domain]))
        S_pred = float(np.sum(nu1[in_domain] * phi1_pred[in_domain] + nu2[in_domain] * phi2_pred[in_domain]))
        eps = 1e-30
        if abs(S_ref) < 1e-20:
            raise RuntimeError("S_ref 过小（几乎为 0），请检查 CSV 是否包含燃料区点、phi 列是否正确、或裂变参数是否正确。")
        scale = S_ref / (S_pred + eps)
        print(f"[信息] 裂变源：S_ref={S_ref:.6e}, S_pred={S_pred:.6e}, scale=S_ref/S_pred={scale:.6e}")
    else:
        S_ref = 1.0
        scale = 1.0
        print("[信息] 已关闭裂变源归一化：直接比较模型输出与参考解。")

    phi1_pred_scaled = phi1_pred * scale
    phi2_pred_scaled = phi2_pred * scale

    eps = 1e-30
    phi1_ref_n = phi1_ref / (S_ref + eps)
    phi2_ref_n = phi2_ref / (S_ref + eps)
    phi1_pred_n = phi1_pred_scaled / (S_ref + eps)
    phi2_pred_n = phi2_pred_scaled / (S_ref + eps)

    rel_eps_1 = REL_EPS_FACTOR * float(np.nanmax(np.abs(phi1_ref_n[in_domain]))) + 1e-30
    rel_eps_2 = REL_EPS_FACTOR * float(np.nanmax(np.abs(phi2_ref_n[in_domain]))) + 1e-30

    # ---- 7.6 区域统计：材料区自动生成，fuel/non_fuel 自动按裂变截面判断 ----
    masks, regions, _ = build_region_masks(mat_ids_np, nu1_map, nu2_map)

    rows = []
    metric_cols = ["L2_rel", "MSE", "RMSE", "MAE", "MaxRelErr", "MeanRelErr", "R2"]
    for reg in regions:
        m = masks[reg] & in_domain
        npts = int(np.sum(m))
        if npts == 0:
            rows.append(dict(region=reg, N=0, group="phi1", **{k: np.nan for k in metric_cols}))
            rows.append(dict(region=reg, N=0, group="phi2", **{k: np.nan for k in metric_cols}))
            continue

        met1 = compute_metrics(phi1_ref_n[m], phi1_pred_n[m], rel_eps=rel_eps_1)
        met2 = compute_metrics(phi2_ref_n[m], phi2_pred_n[m], rel_eps=rel_eps_2)
        rows.append(dict(region=reg, N=npts, group="phi1", **met1))
        rows.append(dict(region=reg, N=npts, group="phi2", **met2))

    result = pd.DataFrame(rows)
    pd.set_option("display.width", 200)
    pd.set_option("display.max_columns", 50)
    print("\n===== 归一化后误差指标（按区域）=====")
    print(result.to_string(index=False, float_format=lambda v: f"{v:.6e}" if isinstance(v, float) and np.isfinite(v) else str(v)))

    if SAVE_METRICS_CSV:
        os.makedirs(OUT_DIR, exist_ok=True)
        out_csv = os.path.join(OUT_DIR, f"{_safe_name(case_name)}_metrics.csv")
        result.to_csv(out_csv, index=False, encoding="utf-8-sig")
        print(f"[已保存] 误差指标 CSV：{out_csv}")

    # ---- 7.7 自动剖面线图 ----
    if PLOT_PROFILE:
        p0, p1 = infer_profile_line_from_points(x, y, in_domain)
        print(f"[信息] 剖面线端点：p0={p0}, p1={p1}")
        try:
            plot_profile_line(
                mod, model,
                x_ref_all=x,
                y_ref_all=y,
                phi1_ref_all=phi1_ref,
                phi2_ref_all=phi2_ref,
                p0=p0,
                p1=p1,
                S_ref=S_ref,
                scale=scale,
                case_name=case_name,
                save_fig=SAVE_FIG,
                out_dir=OUT_DIR,
                n_line=PROFILE_N,
            )
        except Exception as e:
            print(f"[提示] 画剖面线图时出错（不影响数值统计）：{e}")

    # ---- 7.8 全域误差图 ----
    try:
        plot_global_error_maps(
            x=x, y=y, in_domain=in_domain,
            phi1_ref_n=phi1_ref_n, phi2_ref_n=phi2_ref_n,
            phi1_pred_n=phi1_pred_n, phi2_pred_n=phi2_pred_n,
            rel_eps_1=rel_eps_1, rel_eps_2=rel_eps_2,
            case_name=case_name,
        )
    except Exception as e:
        print(f"[提示] 画全域偏差分布图时出错（不影响数值统计）：{e}")


if __name__ == "__main__":
    main()


[Run] OUTPUT_DIR = C:\Users\Administrator\Desktop\run_20260511-152744-pid5672
[Run] LOG_PATH   = C:\Users\Administrator\Desktop\run_20260511-152744-pid5672\train.log
[Run] START_TIME = 2026-05-11 15:27:44
TRUE_X_IF: [24.0, 56.0]
TRUE_Y_IF: [24.0, 56.0]
[信息] 已从 1.py 构建/更新 TRUE_X_IF、TRUE_Y_IF。
[信息] 模型加载完成。
[信息] CSV 按列名读取：x=% X, y=Y, phi1=phi1 (1) @ 1.095 rad/s, phi2=phi2 (1) @ 1.095 rad/s
[信息] CSV 读取完成，共 12818 个点。
[信息] 自动识别到材料编号：[1, 2, 3]
[信息] 已从 1.py 的 RAW_MATERIALS 中读取裂变参数。
[信息] 裂变源：S_ref=2.084658e+04, S_pred=4.144598e+04, scale=S_ref/S_pred=5.029818e-01

===== 归一化后误差指标（按区域）=====
region     N group       L2_rel          MSE         RMSE          MAE    MaxRelErr   MeanRelErr           R2
  mat1  2020  phi1 1.322434e-03 1.089114e-14 1.043606e-07 8.520505e-08 3.870995e-03 9.206050e-04 9.999722e-01
  mat1  2020  phi2 1.410172e-03 5.874970e-17 7.664835e-09 6.161611e-09 8.573879e-03 8.388577e-04 9.999531e-01
  mat2  3109  phi1 1.624911e-03 2.576511e-14 1.605151e-07 1.437366e-07 3.983452e-03

C:\Users\Administrator\AppData\Local\Temp\ipykernel_5672\1475995361.py:762: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


[已保存] 全域偏差分布图：./eval_plots\RTXpinn_global_error_maps.png


C:\Users\Administrator\AppData\Local\Temp\ipykernel_5672\1475995361.py:858: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
